# Pedir datos a la API por rango de tiempo

Prueba cómo se piden lecturas entre dos horas con `fetch(start, end)` de `api_client.py`.

- Con `USAR_API_REAL = False` lee los datos simulados (`SimulatedSource`), que se comportan igual que la API.
- Con `USAR_API_REAL = True` llama a la API real (`ApiSource`). Para eso hace falta la IP de la Raspberry Pi en `config.yaml` y estar en la misma red.

Reglas del contrato (`platform/docs/api/openapi.yaml`): horas en Europe/Amsterdam, los dos extremos incluidos, como máximo 24 h por petición y 404 si ningún sensor tiene datos en ese rango.

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import yaml

# Buscar la carpeta del repositorio (la que tiene config.yaml) y añadir src
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("No se encuentra config.yaml: abre el notebook dentro del repositorio")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from occupancy.api_client import ApiSource, SimulatedSource, format_time

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

## Paso 1: elegir de dónde vienen los datos

Las dos fuentes tienen el mismo método `fetch(start, end)` y devuelven lo mismo: el campo `data` de la respuesta, o `None` si no hay datos.

In [2]:
USAR_API_REAL = True 

if USAR_API_REAL:
    source = ApiSource(config)
    print("API real:", source.url)
else:
    source = SimulatedSource(ROOT / "data" / "simulated" / "readings.json")
    inicio_datos, fin_datos = source.time_span()
    print("Datos simulados desde", format_time(inicio_datos), "hasta", format_time(fin_datos))

API real: http://192.168.0.181:8080/api/v1/readings/designlab/2


## Paso 2: elegir el rango

Escribe el inicio y el fin como texto, `"AAAA-MM-DD HH:MM"` o `"AAAA-MM-DD HH:MM:SS"`, en hora de Ámsterdam. Con los datos simulados, el rango tiene que caer dentro de lo que imprime el paso 1.

Si prefieres pedir los últimos minutos hasta ahora (útil con la API real), pon `ULTIMOS_MINUTOS` a un número y se ignoran `INICIO` y `FIN`.

In [3]:
from datetime import timedelta

INICIO = "2026-10-20 10:00"
FIN = "2026-10-20 10:05"
ULTIMOS_MINUTOS = 15   # por ejemplo 15 para pedir los últimos 15 minutos; None para usar INICIO y FIN


def leer_hora(texto):
    """Convierte 'AAAA-MM-DD HH:MM' o 'AAAA-MM-DD HH:MM:SS' en datetime."""
    for formato in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try:
            return datetime.strptime(texto.strip(), formato)
        except ValueError:
            pass
    raise ValueError(f"Hora no válida: {texto!r}. Usa 'AAAA-MM-DD HH:MM' o 'AAAA-MM-DD HH:MM:SS'")


if ULTIMOS_MINUTOS is not None:
    end = datetime.now()
    start = end - timedelta(minutes=ULTIMOS_MINUTOS)
else:
    start = leer_hora(INICIO)
    end = leer_hora(FIN)

# Comprobaciones antes de llamar
if end <= start:
    raise ValueError("FIN tiene que ser después de INICIO")
if end - start > timedelta(hours=24):
    print("AVISO: más de 24 h; la API real responderá con error 400")
if not USAR_API_REAL and (end < inicio_datos or start > fin_datos):
    print("AVISO: el rango está fuera de los datos simulados; fetch devolverá None")

print("start_time =", format_time(start))
print("end_time   =", format_time(end))
print("duración   =", end - start)

start_time = 2026-11-04 15:38:46.590
end_time   = 2026-11-04 15:53:46.590
duración   = 0:15:00


## Paso 3: pedir los datos

In [4]:
data = source.fetch(start, end)

if data is None:
    if USAR_API_REAL:
        print("La API respondió 404: no hay datos de", config["api"]["building"], "/", config["api"]["room"],
              "entre", format_time(start), "y", format_time(end))
    else:
        print("No hay datos simulados en ese rango")
else:
    print("Lecturas por sensor:")
    for sensor, readings in data.items():
        print(f"  {sensor:<15} {len(readings)}")

Lecturas por sensor:
  bleAdverts      90
  busy_level      90
  co2Equivalent   90
  humidity        90
  iaq             90
  mic_vol         90
  pirs            90
  pressure        90
  temperature     90
  voc             90
  wifiPackets     90


Con una lectura cada 10 s, 5 minutos dan 31 lecturas: 30 intervalos más 1, porque los dos extremos se incluyen.

## Paso 4: ver un sensor normal como tabla

Cada lectura es un diccionario con `timestamp` y `value`.

In [7]:
sensor = "temperature"
if data is None:
    print("No hay datos: cambia el rango en el paso 2")
else:
    display(pd.DataFrame(data[sensor]).head())

,timestamp,value
0,2026-11-04 15:38:50.270,24.04
1,2026-11-04 15:39:00.257,24.05
2,2026-11-04 15:39:10.307,24.05
3,2026-11-04 15:39:20.327,24.06
4,2026-11-04 15:39:30.372,24.05


## Paso 5: ver los PIR

Su estructura es distinta: cada lectura trae `values`, una lista con el `ID` y la `activity` (0 a 100) de cada PIR.

In [8]:
if data is None:
    print("No hay datos: cambia el rango en el paso 2")
else:
    print(data["pirs"][:2])

[{'timestamp': '2026-11-04 15:38:50.270', 'values': [{'ID': 1, 'activity': 50}, {'ID': 2, 'activity': 100}, {'ID': 3, 'activity': 50}]}, {'timestamp': '2026-11-04 15:39:00.257', 'values': [{'ID': 1, 'activity': 50}, {'ID': 2, 'activity': 60}, {'ID': 3, 'activity': 40}]}]


## Paso 6: comprobar los casos límite

- Un rango sin datos devuelve `None`.
- Con la API real, un rango de más de 24 h devuelve error 400 y se ve el mensaje de la API.

In [9]:
# Rango sin datos (otro día)
print(source.fetch(datetime(2026, 10, 21, 10, 0), datetime(2026, 10, 21, 10, 5)))

None


In [11]:
# Solo con la API real: más de 24 h
if USAR_API_REAL:
    try:
        source.fetch(datetime(2026, 11, 3, 10, 0), datetime(2026, 11, 4, 12, 0))
    except RuntimeError as error:
        print(error)
else:
    print("Esta prueba solo tiene sentido con la API real")

The API answered 400: Time range must be between 0 and 24 hours.


In [12]:
import json
import time
import paho.mqtt.client as mqtt

mensajes = []

def al_recibir(client, userdata, msg):
    mensajes.append((msg.topic, msg.payload.decode(errors="replace")))

cliente = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
cliente.on_message = al_recibir
cliente.connect("192.168.0.181", 1883)
cliente.subscribe("#")           # todos los topics
cliente.loop_start()
time.sleep(30)                   # escucha 30 segundos
cliente.loop_stop()
cliente.disconnect()

print("Mensajes recibidos:", len(mensajes))
for topic, payload in mensajes[:5]:
    print("\nTOPIC:", topic)
    print("PAYLOAD:", payload[:600])

Mensajes recibidos: 3

TOPIC: sensors/designlab/2
PAYLOAD: {"temperature":24.42,"humidity":34.49,"pressure":1027.98,"iaq":76.45,"voc":0.65,"co2Equivalent":646.22,"pirs":[{"ID":1,"activity":0},{"ID":2,"activity":100},{"ID":3,"activity":50}],"mic_vol":-32.94,"wifiPackets":363,"bleAdverts":148,"busy_level":3,"timestamp":1793804094338}

TOPIC: sensors/designlab/2
PAYLOAD: {"temperature":24.42,"humidity":34.45,"pressure":1027.98,"iaq":76.13,"voc":0.65,"co2Equivalent":645.66,"pirs":[{"ID":1,"activity":0},{"ID":2,"activity":100},{"ID":3,"activity":40}],"mic_vol":-35.29,"wifiPackets":372,"bleAdverts":158,"busy_level":3,"timestamp":1793804102966}

TOPIC: sensors/designlab/2
PAYLOAD: {"temperature":24.42,"humidity":34.43,"pressure":1027.98,"iaq":76.37,"voc":0.65,"co2Equivalent":646.07,"pirs":[{"ID":1,"activity":10},{"ID":2,"activity":100},{"ID":3,"activity":70}],"mic_vol":-32.93,"wifiPackets":408,"bleAdverts":182,"busy_level":3,"timestamp":1793804112986}
